# Rubik AI: training DeepCubeA di Google Colab (GPU)

DeepCubeA belajar memperkirakan **berapa langkah lagi menuju selesai** (value network),
dilatih dengan *Deep Approximate Value Iteration*, lalu menyelesaikan kubus dengan **batch weighted A\***.
Arsitekturnya sama dengan EfficientCube besar (14,7 juta parameter), supaya perbandingannya adil.

Alurnya sama dengan notebook sebelumnya: jalankan sel dari atas ke bawah. Checkpoint tersimpan di
Google Drive, jadi kalau Colab terputus, jalankan ulang sel 1–4 lalu sel 5 untuk melanjutkan.

## 1. Pastikan GPU aktif

Menu **Runtime → Ubah jenis runtime → T4 GPU → Simpan**.

In [ ]:
!nvidia-smi

## 2. Hubungkan Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/Rubik_AI_colab"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Folder penyimpanan:", DRIVE_DIR)

## 3. Ambil kode dari GitHub

Buat token baru seperti sebelumnya (Fine-grained, hanya repository **Rubik_AI**, **Contents: Read-only**).
Salin token, **jangan di-screenshot**, lalu tempel di kotak yang muncul.

In [ ]:
import getpass, os, subprocess

REPO = "HaiqalGITuru/Rubik_AI"
WORKDIR = "/content/Rubik_AI"

if not os.path.exists(WORKDIR):
    token = getpass.getpass("Tempel GitHub token: ")
    r = subprocess.run(["git", "clone", f"https://{token}@github.com/{REPO}.git", WORKDIR],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, "***"))
    subprocess.run(["git", "-C", WORKDIR, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"])
    del token
else:
    print("Kode sudah ada di", WORKDIR)

os.chdir(WORKDIR)
!ls

## 4. Pengaturan training

- Setiap langkah: 300 scramble × 30 gerakan = **9.000 posisi**, dan untuk setiap posisi jaringan target
  menilai ke-12 anaknya (108.000 evaluasi per langkah).
- **Target network diperbarui setiap 500 langkah**, jadi 50.000 langkah = 100 kali pembaruan. Ini penting:
  perkiraan jarak hanya bisa tumbuh sekitar +1 per pembaruan, sedangkan kubus teracak penuh berjarak
  sampai sekitar 26 gerakan seperempat.
- Setelah sel 5 berjalan beberapa menit, lihat angka `sisa ~... mnt`. Kalau totalnya jauh melebihi
  ~5 jam (setara waktu training EfficientCube besar), hentikan, perkecil `STEPS`, dan mulai ulang
  **dengan menghapus file checkpoint di Drive terlebih dahulu**. Jangan ubah `STEPS` di tengah training.

In [ ]:
MODEL_PATH = f"{DRIVE_DIR}/deepcubea_large.pt"
STEPS = 50_000
SCRAMBLES = 300
UPDATE_EVERY = 500
print("Checkpoint:", MODEL_PATH, "| ada:", os.path.exists(MODEL_PATH))

## 5. Training

Perhatikan kolom **target rata2** dan **maks**: nilainya naik perlahan seiring pembaruan target network,
sampai stabil di sekitar jarak sebenarnya. Ini tanda DeepCubeA sedang "belajar menghitung jarak".

In [ ]:
!python -m scripts.train_deepcubea --device cuda --hidden1 5000 --hidden2 1000 --blocks 4 --scrambles {SCRAMBLES} --max-depth 30 --steps {STEPS} --update-every {UPDATE_EVERY} --cosine --log-every 200 --eval-every 2000 --save {MODEL_PATH} --resume

## 6. Grafik training

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv(MODEL_PATH.replace(".pt", "_log.csv"))
ev = log.dropna(subset=["solve_d20"])

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].plot(log.step, log.loss, color="#E8762D"); ax[0].set_title("Loss (MSE)"); ax[0].set_xlabel("langkah")
ax[1].plot(log.step, log.target_mean, color="#E8762D", label="rata-rata")
ax[1].plot(log.step, log.target_max, color="#E8762D", alpha=0.4, label="maksimum")
ax[1].set_title("Perkiraan jarak (target DAVI)"); ax[1].set_xlabel("langkah"); ax[1].legend(frameon=False)
for d, c in [(10, "#F2C2A0"), (15, "#EE9A62"), (20, "#E8762D")]:
    ax[2].plot(ev.step, ev[f"solve_d{d}"] * 100, "o-", color=c, ms=3, label=f"jarak {d}")
ax[2].set_title("Tingkat berhasil (evaluasi ringan)"); ax[2].set_ylabel("%"); ax[2].legend(frameon=False)
for a in ax: a.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/grafik_training_deepcubea.png", dpi=150)
plt.show()

## 7. Ekspor model ringan (~29 MB)

In [ ]:
INFER_PATH = f"{DRIVE_DIR}/deepcubea_large_infer.pt"
!python -m scripts.export_model {MODEL_PATH} {INFER_PATH}

## 8. Uji pada 100 kubus yang sama

Batch weighted A\* dengan bobot 0,6 (sesuai pengaturan yang dilaporkan paper DeepCubeA) dan ukuran batch
N = 100, 1.000, dan 10.000. Hasil langsung disalin ke Drive.

In [ ]:
import shutil
from IPython.display import Image, display

!python -m scripts.beam_experiment --model {INFER_PATH} --device cuda --solver astar --beams 100 1000 10000 --name deepcubea_large

shutil.copytree("results/deepcubea_large", f"{DRIVE_DIR}/deepcubea_large", dirs_exist_ok=True)
display(Image("results/deepcubea_large/ringkasan.png"))

## 9. Perbandingan EfficientCube vs DeepCubeA (semua di GPU yang sama)

Memakai hasil EfficientCube yang sudah ada di repository (`results/beam_experiment_*`).

In [ ]:
!python -m scripts.compare_results beam_experiment_small_gpu:"EfficientCube kecil" beam_experiment_large:"EfficientCube besar" deepcubea_large:"DeepCubeA besar"

shutil.copytree("results/perbandingan", f"{DRIVE_DIR}/perbandingan", dirs_exist_ok=True)
display(Image("results/perbandingan/perbandingan.png"))

## 10. Bawa hasilnya ke laptop

Dari folder **Rubik_AI_colab** di Google Drive, unduh:

- `deepcubea_large_infer.pt` → `Rubik_AI/models/`
- folder `deepcubea_large` dan `perbandingan` → `Rubik_AI/results/` (buka zip, lalu **seret foldernya**, jangan "Extract All")
- `deepcubea_large_log.csv` dan `grafik_training_deepcubea.png` → `Rubik_AI/results/colab/`

Setelah selesai, hapus token GitHub yang dipakai untuk Colab.